In [20]:
import pandas as pd
import statsmodels.api as sm

# 1. Cargar la base
df = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_turnover_2022_con_delitos.xlsx")


In [21]:

# 2. Definir variables (ajusta estos nombres si es necesario)
y_col = "delitos_suma_2023_2025"   # outcome (criminalidad)
D_col = "turnover_candidato"       # tratamiento (turnover)
Z_col = "orden_aparicion"          # instrumento (posición en la cédula)

# 3. Sub-muestra sólo con las columnas necesarias y sin NA
df_iv = df[[y_col, D_col, Z_col]].dropna()

y = df_iv[y_col]   # outcome
D = df_iv[D_col]   # tratamiento
Z = df_iv[Z_col]   # instrumento


In [22]:
# ============================
# PRIMERA ETAPA: D ~ Z
# ============================
X1 = sm.add_constant(Z)  # agrega intercepto
first_stage = sm.OLS(D, X1).fit()

print("=== PRIMERA ETAPA: efecto del instrumento en el tratamiento ===")
print(first_stage.summary())

# Predicciones de D (D_hat)
df_iv["D_hat"] = first_stage.fittedvalues


=== PRIMERA ETAPA: efecto del instrumento en el tratamiento ===
                            OLS Regression Results                            
Dep. Variable:     turnover_candidato   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     2.249
Date:                Sun, 30 Nov 2025   Prob (F-statistic):              0.134
Time:                        13:38:27   Log-Likelihood:                 1538.6
No. Observations:                 930   AIC:                            -3073.
Df Residuals:                     928   BIC:                            -3063.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------

In [23]:
# ============================
# SEGUNDA ETAPA: y ~ D_hat
# ============================
X2 = sm.add_constant(df_iv["D_hat"])
second_stage = sm.OLS(y, X2).fit()

print("\n=== SEGUNDA ETAPA (2SLS): efecto causal de turnover en criminalidad ===")
print(second_stage.summary())

# Coeficiente IV de interés (sobre D_hat)
beta_iv = second_stage.params["D_hat"]
print("\nCoeficiente IV (aprox. efecto causal de turnover sobre criminalidad):", beta_iv)



=== SEGUNDA ETAPA (2SLS): efecto causal de turnover en criminalidad ===
                              OLS Regression Results                              
Dep. Variable:     delitos_suma_2023_2025   R-squared:                       0.032
Model:                                OLS   Adj. R-squared:                  0.031
Method:                     Least Squares   F-statistic:                     31.14
Date:                    Sun, 30 Nov 2025   Prob (F-statistic):           3.15e-08
Time:                            13:38:37   Log-Likelihood:                -8983.4
No. Observations:                     930   AIC:                         1.797e+04
Df Residuals:                         928   BIC:                         1.798e+04
Df Model:                               1                                         
Covariance Type:                nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------

In [24]:
tabla_resumen = pd.DataFrame({
    "etapa": ["primera", "segunda"],
    "variable_clave": [Z_col, "D_hat"],
    "coef": [
        first_stage.params[Z_col],
        second_stage.params["D_hat"]
    ],
    "se_robusta": [
        first_stage.bse[Z_col],
        second_stage.bse["D_hat"]
    ],
    "pvalue": [
        first_stage.pvalues[Z_col],
        second_stage.pvalues["D_hat"]
    ],
    "R2": [
        first_stage.rsquared,
        second_stage.rsquared
    ],
    "N": [
        int(first_stage.nobs),
        int(second_stage.nobs)
    ]
})

print("\n=== TABLA RESUMEN (coeficientes clave) ===")
print(tabla_resumen)


=== TABLA RESUMEN (coeficientes clave) ===
     etapa   variable_clave           coef    se_robusta        pvalue  \
0  primera  orden_aparicion       0.001213      0.000809  1.340304e-01   
1  segunda            D_hat  304951.653326  54648.831453  3.152675e-08   

         R2    N  
0  0.002418  930  
1  0.032465  930  
